# Colab Z-Image Turbo + ngrok API

啟動 ComfyUI，並用 ngrok 暴露 Web UI 與 ComfyUI API。

In [ ]:
# 1. Install ComfyUI + dependencies
%cd /content
![ -d ComfyUI ] || git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip -q install -r requirements.txt xformers pyngrok

In [ ]:
# 2. Download Z-Image Turbo models
%cd /content/ComfyUI
!mkdir -p models/diffusion_models models/text_encoders models/vae

!test -f models/diffusion_models/z_image_turbo_nvfp4.safetensors || wget -q --show-progress -O models/diffusion_models/z_image_turbo_nvfp4.safetensors https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/diffusion_models/z_image_turbo_nvfp4.safetensors
!test -f models/text_encoders/qwen_3_4b_fp4_mixed.safetensors || wget -q --show-progress -O models/text_encoders/qwen_3_4b_fp4_mixed.safetensors https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b_fp4_mixed.safetensors
!test -f models/vae/ae.safetensors || wget -q --show-progress -O models/vae/ae.safetensors https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors

In [ ]:
# 3. Start ngrok tunnel
from getpass import getpass
from pyngrok import ngrok

try:
    from google.colab import userdata
    token = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    token = None

if not token:
    token = getpass("NGROK_AUTHTOKEN: ")

ngrok.set_auth_token(token)
ngrok.kill()

public_url = ngrok.connect(8188, "http").public_url
print("ComfyUI Web UI:", public_url)
print("ComfyUI API:", public_url + "/prompt")
print("History API:", public_url + "/history/{prompt_id}")

In [ ]:
# 4. Launch ComfyUI
%cd /content/ComfyUI
!python main.py --listen 0.0.0.0 --port 8188 --dont-upcast-attention --lowvram --enable-cors-header '*'